# **Setup**

In [17]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

Repo already exists — pulling latest changes
Already up to date.


In [18]:
import numpy as np
import optuna

from Challenge.paths import load_cv_folds
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender

# **Load Data**

In [19]:
# Load datasets
folds = load_cv_folds(k=5)

# **Hyperparameter search**

In [45]:
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython

optimizer = ModelOptimizer("MatrixFactorization_WARP")

STUDY_NAME = MatrixFactorization_WARP_Cython.RECOMMENDER_NAME + "_v4"

In [46]:
URM_train, URM_val = folds[0]

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "batch_size": 1024,
        "sgd_mode": 'adagrad',
        "use_bias": False,
        "num_factors": 150,
        "learning_rate": 0.07149408001967965,
        "WARP_neg_item_attempts": 50,
        "user_reg": 0.009339267088021491,
        "positive_reg": 0.009339267088021491,
        "negative_reg": 0.0,
        "epochs": optuna_trial.suggest_categorical("epochs", [1250, 1500, 1750, 2000]),
    }
    
    # Train the recommender
    recommender_instance = MatrixFactorization_WARP_Cython(URM_train)
    recommender_instance.fit(**params)
    
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_val)
        
    # Log folds performance
    optimizer.log_folds([score], params)

    return score

In [47]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=5
)

[I 2025-11-28 21:25:09,532] A new study created in RDB with name: MatrixFactorization_WARP_Cython_Recommender_v4


  0%|          | 0/5 [00:00<?, ?it/s]

MF_WARP: Processed 27648 (100.0%) in 1.10 sec. MSE loss 8.12E-02. Sample per second: 25155
MF_WARP: Epoch 1 of 1500. Elapsed time 0.31 sec
MF_WARP: Processed 27648 (100.0%) in 0.43 sec. MSE loss 1.44E-01. Sample per second: 64368
MF_WARP: Epoch 2 of 1500. Elapsed time 0.64 sec
MF_WARP: Processed 27648 (100.0%) in 0.74 sec. MSE loss 1.94E-01. Sample per second: 37359
MF_WARP: Epoch 3 of 1500. Elapsed time 0.95 sec
MF_WARP: Processed 27648 (100.0%) in 1.07 sec. MSE loss 2.33E-01. Sample per second: 25917
MF_WARP: Epoch 4 of 1500. Elapsed time 1.28 sec
MF_WARP: Processed 27648 (100.0%) in 0.32 sec. MSE loss 2.80E-01. Sample per second: 86648
MF_WARP: Epoch 5 of 1500. Elapsed time 1.53 sec
MF_WARP: Processed 27648 (100.0%) in 0.52 sec. MSE loss 3.40E-01. Sample per second: 53233
MF_WARP: Epoch 6 of 1500. Elapsed time 1.73 sec
MF_WARP: Processed 27648 (100.0%) in 0.71 sec. MSE loss 4.46E-01. Sample per second: 38885
MF_WARP: Epoch 7 of 1500. Elapsed time 1.92 sec
MF_WARP: Processed 27648 (1

Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  8.15it/s]


[I 2025-11-28 21:30:41,619] Trial 0 finished with value: 0.198120126660741 and parameters: {'epochs': 1500}. Best is trial 0 with value: 0.198120126660741.
MF_WARP: Processed 27648 (100.0%) in 1.08 sec. MSE loss 8.20E-02. Sample per second: 25564
MF_WARP: Epoch 1 of 1500. Elapsed time 0.25 sec
MF_WARP: Processed 27648 (100.0%) in 0.25 sec. MSE loss 1.49E-01. Sample per second: 110059
MF_WARP: Epoch 2 of 1500. Elapsed time 0.42 sec
MF_WARP: Processed 27648 (100.0%) in 0.42 sec. MSE loss 1.91E-01. Sample per second: 65212
MF_WARP: Epoch 3 of 1500. Elapsed time 0.59 sec
MF_WARP: Processed 27648 (100.0%) in 0.60 sec. MSE loss 2.32E-01. Sample per second: 46200
MF_WARP: Epoch 4 of 1500. Elapsed time 0.76 sec
MF_WARP: Processed 27648 (100.0%) in 0.77 sec. MSE loss 2.78E-01. Sample per second: 35687
MF_WARP: Epoch 5 of 1500. Elapsed time 0.94 sec
MF_WARP: Processed 27648 (100.0%) in 0.97 sec. MSE loss 3.40E-01. Sample per second: 28505
MF_WARP: Epoch 6 of 1500. Elapsed time 1.14 sec
MF_WARP: 

KeyboardInterrupt: 

In [30]:
optuna.visualization.plot_optimization_history(optuna_study)

In [31]:
optuna.visualization.plot_param_importances(optuna_study)

In [32]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE